# Amazon ML Challenge 2026 — Day 1: EDA & Baseline Pipeline

**Official Evaluation Metric:** Macro-averaged $F_{0.5}$ (Precision-weighted entity resolution: $\beta=0.5$)

### Memory-Optimized Architecture
* Fully loads **Source 1** (2.2M rows) and **Ground Truth** (2.2M rows) into RAM (~320 MB total).
* Streams **Source 2** (~5.0M rows) and **Source 3** (~5.3M rows) in `250,000`-row chunks.
* Candidate generation is **targeted**: only candidates matching query keys are indexed, preventing OOM crashes.
* Test submission generates valid `test_source2` and `test_source3` candidate pairs.

In [ ]:
import os
import sys
import re
import time
import subprocess
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# 1. Locate directories reliably
POSSIBLE_DIRS = ['../data/dataset/train', 'data/dataset/train', '../dataset/train', 'dataset/train']
TRAIN_DIR = next((d for d in POSSIBLE_DIRS if os.path.exists(d)), None)
if not TRAIN_DIR:
    raise FileNotFoundError('Could not locate train dataset directory.')

POSSIBLE_TEST_DIRS = ['../data/dataset/test', 'data/dataset/test', '../dataset/test', 'dataset/test']
TEST_DIR = next((d for d in POSSIBLE_TEST_DIRS if os.path.exists(d)), None)

# Ensure output directory exists
OUT_DIR = '../output' if os.path.exists('../output') or os.path.exists('../src') else 'output'
os.makedirs(OUT_DIR, exist_ok=True)

print(f'Train directory : {os.path.abspath(TRAIN_DIR)}')
print(f'Test directory  : {os.path.abspath(TEST_DIR) if TEST_DIR else "Not Found"}')
print(f'Output directory: {os.path.abspath(OUT_DIR)}')
sys.path.insert(0, os.path.abspath('../src'))
sys.path.insert(0, os.path.abspath('src'))

## 1. Quick Sample Inspection (`nrows=1000`)
Fast inspection to verify columns and data formats before loading.

In [ ]:
s1_sample = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source1.tsv'), sep='\t', nrows=1000)
s2_sample_head = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source2.tsv'), sep='\t', nrows=1000)
s3_sample_head = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source3.tsv'), sep='\t', nrows=1000)
gt_sample = pd.read_csv(os.path.join(TRAIN_DIR, 'train_ground_truth.tsv'), sep='\t', nrows=1000)

print('Source 1 Columns:', s1_sample.columns.tolist())
print('Source 2 Columns:', s2_sample_head.columns.tolist())
print('Source 3 Columns:', s3_sample_head.columns.tolist())
print('Ground Truth Columns:', gt_sample.columns.tolist())
print('\n=== Source 1 Sample (3 rows) ===')
print(s1_sample.head(3))
print('\n=== Ground Truth Sample (3 rows) ===')
print(gt_sample.head(3))

## 2. Memory-Safe Loading Strategy

**The Rule:** Do **not** load full S2 (~5.0M rows) and S3 (~5.3M rows) simultaneously into RAM.
* Fully load **Source 1** (2.2M rows ~ 200 MB) and **Ground Truth** (2.2M rows ~ 120 MB).
* Stream **Source 2** and **Source 3** in chunks of `250,000` rows.
* During streaming, collect total rows, missing counts, country frequencies, and matched samples for inspection.

In [ ]:
dtype_spec = {
    'entity_id': 'string',
    'business_name': 'string',
    'business_address': 'string',
    'country': 'category'
}

t_start = time.time()

# 1. Load Source 1
print('Loading Source 1 into memory...')
s1 = pd.read_csv(os.path.join(TRAIN_DIR, 'train_source1.tsv'), sep='\t', dtype=dtype_spec)
print(f'  Source 1 loaded: {len(s1):,} rows')

# 2. Load Ground Truth
print('Loading Ground Truth into memory...')
gt = pd.read_csv(
    os.path.join(TRAIN_DIR, 'train_ground_truth.tsv'), 
    sep='\t', 
    dtype={'source1_entity_id': 'string', 'matched_entity_ids': 'string'}
)
print(f'  Ground Truth loaded: {len(gt):,} rows')

# Pre-identify candidate IDs from the first 20 GT matches to capture during streaming
gt_inspect_targets = set()
for mid_str in gt['matched_entity_ids'].dropna().head(50):
    for m in mid_str.split(','):
        m_clean = m.strip()
        if m_clean: gt_inspect_targets.add(m_clean)

# 3. Stream Source 2 in chunks of 250,000 rows
print('\nStreaming Source 2 in chunks of 250,000...')
s2_total_rows = 0
s2_missing = None
s2_country_counts = Counter()
s2_matched_samples = []

for i, chunk in enumerate(pd.read_csv(os.path.join(TRAIN_DIR, 'train_source2.tsv'), sep='\t', dtype=dtype_spec, chunksize=250_000)):
    s2_total_rows += len(chunk)
    s2_missing = chunk.isnull().sum() if s2_missing is None else s2_missing + chunk.isnull().sum()
    s2_country_counts.update(chunk['country'].value_counts().to_dict())
    # Grab inspected rows
    hits = chunk[chunk['entity_id'].isin(gt_inspect_targets)]
    if not hits.empty:
        s2_matched_samples.append(hits)
    if (i + 1) % 4 == 0 or s2_total_rows > 5_000_000:
        print(f'  Processed {s2_total_rows:,} rows of Source 2...')

s2_sample = pd.concat(s2_matched_samples, ignore_index=True) if s2_matched_samples else pd.DataFrame()
print(f'Source 2 stream complete: {s2_total_rows:,} total rows')

# 4. Stream Source 3 in chunks of 250,000 rows
print('\nStreaming Source 3 in chunks of 250,000...')
s3_total_rows = 0
s3_missing = None
s3_country_counts = Counter()
s3_matched_samples = []

for i, chunk in enumerate(pd.read_csv(os.path.join(TRAIN_DIR, 'train_source3.tsv'), sep='\t', dtype=dtype_spec, chunksize=250_000)):
    s3_total_rows += len(chunk)
    s3_missing = chunk.isnull().sum() if s3_missing is None else s3_missing + chunk.isnull().sum()
    s3_country_counts.update(chunk['country'].value_counts().to_dict())
    hits = chunk[chunk['entity_id'].isin(gt_inspect_targets)]
    if not hits.empty:
        s3_matched_samples.append(hits)
    if (i + 1) % 4 == 0 or s3_total_rows > 5_000_000:
        print(f'  Processed {s3_total_rows:,} rows of Source 3...')

s3_sample = pd.concat(s3_matched_samples, ignore_index=True) if s3_matched_samples else pd.DataFrame()
print(f'Source 3 stream complete: {s3_total_rows:,} total rows')
print(f'\nAll files processed safely in {time.time() - t_start:.2f} seconds without RAM overload!')

## 3. Missing Value Analysis (Full Datasets)
Exact missing value counts across all 12.5M records computed from chunked pass.

In [ ]:
# S1 missing
s1_missing = s1.isnull().sum()
s1_pct = (s1_missing / len(s1) * 100).round(2)
print('=== Source 1 Missing Values (2.2M rows) ===')
print(pd.DataFrame({'Missing': s1_missing, 'Percent (%)': s1_pct}))
print()

# S2 missing
s2_pct = (s2_missing / s2_total_rows * 100).round(2)
print(f'=== Source 2 Missing Values ({s2_total_rows:,} rows) ===')
print(pd.DataFrame({'Missing': s2_missing, 'Percent (%)': s2_pct}))
print()

# S3 missing
s3_pct = (s3_missing / s3_total_rows * 100).round(2)
print(f'=== Source 3 Missing Values ({s3_total_rows:,} rows) ===')
print(pd.DataFrame({'Missing': s3_missing, 'Percent (%)': s3_pct}))

## 4. Country Distribution Analysis (Full Datasets)
Top country distributions across all sources.

In [ ]:
print('=== Source 1 Top Countries ===')
print(s1['country'].value_counts().head(10))
print()

print('=== Source 2 Top Countries ===')
print(pd.Series(s2_country_counts).sort_values(ascending=False).head(10))
print()

print('=== Source 3 Top Countries ===')
print(pd.Series(s3_country_counts).sort_values(ascending=False).head(10))

## 5. Duplicate Records & Entity Key Analysis

In [ ]:
print(f'Source 1 Unique Entity IDs: {s1["entity_id"].nunique():,} / {len(s1):,}')
print(f'Ground Truth Unique S1 IDs: {gt["source1_entity_id"].nunique():,} / {len(gt):,}')
print(f'Source 2 Total Entities   : {s2_total_rows:,}')
print(f'Source 3 Total Entities   : {s3_total_rows:,}')

## 6. Ground Truth Match Count Distribution Analysis
Analysis of singletons (entities with 0 matches) vs multi-match records.

In [ ]:
def parse_matches(val):
    if pd.isna(val) or str(val).strip() == '':
        return []
    return [m.strip() for m in str(val).split(',') if m.strip()]

gt['match_list'] = gt['matched_entity_ids'].apply(parse_matches)
gt['num_matches'] = gt['match_list'].apply(len)

match_counts = gt['num_matches'].value_counts().sort_index()
print('Match count distribution per S1 entity:')
print(match_counts)

zero_matches = (gt['num_matches'] == 0).sum()
total = len(gt)
print(f'\nSingletons (0 matches)  : {zero_matches:,} ({zero_matches / total * 100:.2f}%)')
print(f'Entities with >=1 match : {total - zero_matches:,} ({(total - zero_matches) / total * 100:.2f}%)')
print(f'Max matches for one S1  : {gt["num_matches"].max()}')

## 7. Inspect Real Matched Pairs (Noise Patterns)
Comparing raw names and addresses of true matches.

In [ ]:
s1_idx = s1.set_index('entity_id')
s2_idx = s2_sample.set_index('entity_id') if not s2_sample.empty else pd.DataFrame()
s3_idx = s3_sample.set_index('entity_id') if not s3_sample.empty else pd.DataFrame()

matched_rows = gt[gt['num_matches'] > 0]
printed = 0
for _, row in matched_rows.iterrows():
    s1_id = row['source1_entity_id']
    targets_in_sample = [m for m in row['match_list'] if (not s2_idx.empty and m in s2_idx.index) or (not s3_idx.empty and m in s3_idx.index)]
    if targets_in_sample and s1_id in s1_idx.index:
        s1_r = s1_idx.loc[s1_id]
        print(f'=== Source 1 [{s1_id}] ===')
        print(f'  Name   : {s1_r["business_name"]}')
        print(f'  Address: {s1_r["business_address"]}')
        print(f'  Country: {s1_r["country"]}')
        for mid in targets_in_sample:
            m_r = s2_idx.loc[mid] if (not s2_idx.empty and mid in s2_idx.index) else s3_idx.loc[mid]
            print(f'  -> Matched [{mid}]:')
            print(f'       Name   : {m_r["business_name"]}')
            print(f'       Address: {m_r["business_address"]}')
            print(f'       Country: {m_r["country"]}')
        print('-' * 80)
        printed += 1
        if printed >= 4: break

## 8. Entity-Stratified Validation Split
We split Source 1 into a fast validation set (10,000 entities) to validate Day 1 baseline accuracy.

In [ ]:
np.random.seed(42)
all_s1_ids = gt['source1_entity_id'].values
n_val = 10_000
val_ids = set(np.random.choice(all_s1_ids, size=n_val, replace=False))

val_gt = gt[gt['source1_entity_id'].isin(val_ids)].copy()
val_s1 = s1[s1['entity_id'].isin(val_ids)].copy()
print(f'Validation set created: {len(val_gt):,} Source 1 entities')

## 9. Targeted Chunked Indexing for Validation Split
Extract target keys `(country, norm_name)` for validation entities and stream S2/S3 chunks, keeping **only candidates that actually match validation queries**.

In [ ]:
def clean_name(s):
    if pd.isna(s): return ''
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\b(inc|corp|ltd|llc|gmbh|co|company)\b', '', s)
    return ' '.join(s.split())

val_s1['norm_name'] = [clean_name(x) for x in val_s1['business_name']]

# Build set of validation search keys
val_target_keys = set(zip(val_s1['country'].astype(str), val_s1['norm_name']))
print(f'Validation search keys to match: {len(val_target_keys):,}')

# Stream train S2 and S3, retaining ONLY matching candidates
cand_index = defaultdict(list)
t0 = time.time()

for src_file in ['train_source2.tsv', 'train_source3.tsv']:
    src_path = os.path.join(TRAIN_DIR, src_file)
    for chunk in pd.read_csv(src_path, sep='\t', usecols=['entity_id', 'business_name', 'country'], dtype=dtype_spec, chunksize=500_000):
        chunk['norm_name'] = [clean_name(x) for x in chunk['business_name']]
        keys = list(zip(chunk['country'].astype(str), chunk['norm_name']))
        for cid, k in zip(chunk['entity_id'], keys):
            if k in val_target_keys:
                cand_index[k].append(cid)

print(f'Validation candidate index built in {time.time() - t0:.2f}s!')
print(f'Found candidates for {len(cand_index):,} unique (country, name) validation keys.')

## 10. Baseline Evaluation: Official Macro $F_{0.5}$ Metric

In [ ]:
val_preds = {}
for sid, norm, ctry in zip(val_s1['entity_id'], val_s1['norm_name'], val_s1['country']):
    matches = cand_index.get((str(ctry), norm), [])
    val_preds[sid] = set(matches[:5]) # cap to top 5 candidates

# Compute official Macro F0.5
beta = 0.5
f_scores = []
precisions = []
recalls = []

for _, row in val_gt.iterrows():
    sid = row['source1_entity_id']
    true_set = set(row['match_list'])
    pred_set = val_preds.get(sid, set())
    
    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    
    if len(true_set) == 0 and len(pred_set) == 0:
        f_scores.append(1.0)
        precisions.append(1.0)
        recalls.append(1.0)
    elif len(pred_set) == 0 or len(true_set) == 0:
        f_scores.append(0.0)
        precisions.append(0.0 if len(pred_set) > 0 else 1.0)
        recalls.append(0.0 if len(true_set) > 0 else 1.0)
    else:
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f05 = ((1 + beta**2) * p * r) / ((beta**2 * p) + r) if ((beta**2 * p) + r) > 0 else 0.0
        f_scores.append(f05)
        precisions.append(p)
        recalls.append(r)

print('=== Day 1 Baseline Validation Results (10,000 S1 Entities) ===')
print(f'Macro F0.5      : {np.mean(f_scores):.4f}')
print(f'Macro Precision : {np.mean(precisions):.4f}')
print(f'Macro Recall    : {np.mean(recalls):.4f}')

## 11. Generate Baseline Test Submission Files
Matches test Source 1 entities against **test Source 2 and test Source 3** candidates via fast chunked streaming.

In [ ]:
matching_path = os.path.join(OUT_DIR, 'matching_results.tsv')
candidate_path = os.path.join(OUT_DIR, 'candidate_pairs.tsv')

if not TEST_DIR or not os.path.exists(TEST_DIR):
    print('Test directory not found — skipping test submission generation.')
else:
    print(f'Generating submissions into: {OUT_DIR}')
    # 1. Load test S1
    test_s1 = pd.read_csv(os.path.join(TEST_DIR, 'test_source1.tsv'), sep='\t', usecols=['entity_id', 'business_name', 'country'], dtype=dtype_spec)
    test_s1['norm_name'] = [clean_name(x) for x in test_s1['business_name']]
    test_target_keys = set(zip(test_s1['country'].astype(str), test_s1['norm_name']))
    print(f'Loaded {len(test_s1):,} Test S1 entities. Target keys: {len(test_target_keys):,}')
    
    # 2. Stream test S2 and S3 to find candidates
    test_cand_index = defaultdict(list)
    t_test = time.time()
    for test_src in ['test_source2.tsv', 'test_source3.tsv']:
        p = os.path.join(TEST_DIR, test_src)
        if os.path.exists(p):
            print(f'  Streaming {test_src}...')
            for chunk in pd.read_csv(p, sep='\t', usecols=['entity_id', 'business_name', 'country'], dtype=dtype_spec, chunksize=500_000):
                chunk['norm_name'] = [clean_name(x) for x in chunk['business_name']]
                keys = list(zip(chunk['country'].astype(str), chunk['norm_name']))
                for cid, k in zip(chunk['entity_id'], keys):
                    if k in test_target_keys:
                        test_cand_index[k].append(cid)
    print(f'Test candidate stream finished in {time.time() - t_test:.2f}s!')
    
    # 3. Write submission TSVs (100% of test S1 rows guaranteed)
    matching_rows = []
    candidate_rows = []
    for sid, norm, ctry in zip(test_s1['entity_id'], test_s1['norm_name'], test_s1['country']):
        matched = test_cand_index.get((str(ctry), norm), [])
        pred_str = ','.join(matched[:5]) if matched else ''
        matching_rows.append(f'{sid}\t{pred_str}\n')
        candidate_rows.append(f'{sid}\t{pred_str}\n')
    
    with open(matching_path, 'w', encoding='utf-8') as f:
        f.write('source1_entity_id\tmatched_entity_ids\n')
        f.writelines(matching_rows)
    
    with open(candidate_path, 'w', encoding='utf-8') as f:
        f.write('source1_entity_id\tcandidate_entity_ids\n')
        f.writelines(candidate_rows)
    
    print(f'Saved: {matching_path} ({len(matching_rows):,} rows)')
    print(f'Saved: {candidate_path} ({len(candidate_rows):,} rows)')

## 12. Validate Submission with Official Validator

In [ ]:
validator_candidates = [
    '../data/utils/validate_submission.py',
    'data/utils/validate_submission.py',
    '../utils/validate_submission.py',
    'utils/validate_submission.py'
]
validator_path = next((p for p in validator_candidates if os.path.exists(p)), None)

if not validator_path:
    print('Validator script not found!')
else:
    print(f'Running validator: {os.path.abspath(validator_path)}')
    cmd = [
        sys.executable,
        os.path.abspath(validator_path),
        '--matching', os.path.abspath(matching_path),
        '--candidate', os.path.abspath(candidate_path),
        '--test-dir', os.path.abspath(TEST_DIR)
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.stderr:
        print('Errors:', res.stderr)